In [2]:
import pandas as pd
import sqlite3
import os

# Détection automatique du bon chemin
chemin_db = 'data/processed/supply_chain.db'
if not os.path.exists(chemin_db):
    chemin_db = '../data/processed/supply_chain.db'

if not os.path.exists(chemin_db):
    print("ERREUR : La base de données est introuvable. As-tu bien exécuté le script 02_etl_pipeline.py ?")
else:
    print(f"Connexion réussie à : {chemin_db}")
    conn = sqlite3.connect(chemin_db)

    # Requête SQL pour croiser les clients et les commandes
    query = """
    SELECT 
        o.Shipping_Mode,
        c.Customer_Segment,
        c.Customer_Country,
        o.Late_Delivery_Risk
    FROM Orders o
    JOIN Customers c ON o.Customer_Id = c.Customer_Id
    """
    
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Affiche les 5 premières lignes
    display(df.head())

Connexion réussie à : data/processed/supply_chain.db


,Shipping_Mode,Customer_Segment,Customer_Country,Late_Delivery_Risk
0,Standard Class,Consumer,Puerto Rico,0
1,Standard Class,Consumer,Puerto Rico,1
2,Standard Class,Consumer,EE. UU.,0
3,Standard Class,Home Office,EE. UU.,0
4,Standard Class,Corporate,Puerto Rico,0


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# 1. Encodage : transformation des textes (ex: "Standard Class") en nombres (ex: 0, 1, 2)
le = LabelEncoder()
df['Shipping_Mode'] = le.fit_transform(df['Shipping_Mode'])
df['Customer_Segment'] = le.fit_transform(df['Customer_Segment'])
df['Customer_Country'] = le.fit_transform(df['Customer_Country'])

# 2. Séparation des variables
X = df.drop('Late_Delivery_Risk', axis=1) # Variables explicatives (ce qu'on sait)
y = df['Late_Delivery_Risk']              # Variable cible (ce qu'on veut prédire)

# 3. Découpage : 80% pour l'entraînement, 20% pour le test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Création et entraînement du modèle
print("Entraînement de l'algorithme Random Forest en cours (cela peut prendre quelques secondes)...")
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# 5. Prédictions et évaluation
y_pred = model.predict(X_test)
print("\n--- RÉSULTATS DU MODÈLE ---")
print(f"Précision globale (Accuracy) : {accuracy_score(y_test, y_pred) * 100:.2f} %")
print("\nRapport détaillé :")
print(classification_report(y_test, y_pred))

Entraînement de l'algorithme Random Forest en cours (cela peut prendre quelques secondes)...

--- RÉSULTATS DU MODÈLE ---
Précision globale (Accuracy) : 68.98 %

Rapport détaillé :
              precision    recall  f1-score   support

           0       0.61      0.88      0.72      5952
           1       0.84      0.53      0.65      7199

    accuracy                           0.69     13151
   macro avg       0.73      0.71      0.69     13151
weighted avg       0.74      0.69      0.68     13151

